# Lesson 2: Data Cleaning & Loading


---
## 1. Setup

In [42]:
import sqlite3
import math
import pandas as pd
from datetime import datetime, time, timedelta
from opensearchpy import OpenSearch, helpers
from opensearch_dsl import Document, Keyword, Date, Float, Boolean, Search

pd.set_option('display.width', 300)
pd.set_option('display.max_columns', 20)

In [43]:
con = sqlite3.connect('database.db')

machine_log    = pd.read_sql_query('SELECT * FROM machine_log', con)
maintenance    = pd.read_sql_query('SELECT * FROM maintenance', con)
shift_schedule = pd.read_sql_query('SELECT * FROM shift_schedule', con)

con.close()

print(f'machine_log:    {machine_log.shape}')
print(f'maintenance:    {maintenance.shape}')
print(f'shift_schedule: {shift_schedule.shape}')

machine_log:    (2100, 4)
maintenance:    (707, 4)
shift_schedule: (2318, 6)


---
## 2. Domain: `machine_log`

**Goal:** Convert the two timestamp strings into proper `datetime` objects.

This is the simplest domain — no missing values, no format ambiguity. It's a pure type conversion.

### Dirty Input

In [44]:
machine_log.head(5)

,employee_ID,machine_ID,Time_in,Time_out
0,5ccedc77-3429-479c-acda-4ccb01f35efe,32b55d35-e676-46f1-84c6-9290764b5018,2024-12-02T06:28:00,2024-12-02T12:23:22
1,9fb932d4-f039-4722-96ff-82e389e3995a,bf5cb443-292a-48f3-b713-b4c7ebb7344d,2024-12-02T06:26:00,2024-12-02T09:12:12
2,9fb932d4-f039-4722-96ff-82e389e3995a,14b062ae-88c8-4ad1-aee1-f220fd547512,2024-12-02T14:12:00,2024-12-02T19:07:47
3,5ccedc77-3429-479c-acda-4ccb01f35efe,0d2329d9-fa09-4a46-b673-89669b02a56d,2024-12-02T14:11:00,2024-12-02T17:21:47
4,9fb932d4-f039-4722-96ff-82e389e3995a,f3d2273a-3740-4227-a10f-4d0c5b86c0ef,2024-12-02T22:19:00,2024-12-03T01:33:14


In [45]:
print('dtypes before cleaning:')
print(machine_log.dtypes)
print(f'\nMissing values: {machine_log.isna().sum().sum()}')

dtypes before cleaning:
employee_ID    str
machine_ID     str
Time_in        str
Time_out       str
dtype: object

Missing values: 0


### Cleaning

`pd.to_datetime()` handles ISO 8601 strings like `2025-09-01T06:20:00` automatically. We convert both columns, drop the originals, and rename to snake_case.

In [46]:
machine_log['time_in']  = pd.to_datetime(machine_log['Time_in'])
machine_log['time_out'] = pd.to_datetime(machine_log['Time_out'])

machine_log.drop(columns=['Time_in', 'Time_out'], inplace=True)

# Rename ID columns to snake_case for consistency
machine_log.rename(columns={'employee_ID': 'employee_id', 'machine_ID': 'machine_id'}, inplace=True)

### Clean Output ✓

In [47]:
display(machine_log.head(5))
print()
machine_log.info()

,employee_id,machine_id,time_in,time_out
0,5ccedc77-3429-479c-acda-4ccb01f35efe,32b55d35-e676-46f1-84c6-9290764b5018,2024-12-02 06:28:00,2024-12-02 12:23:22
1,9fb932d4-f039-4722-96ff-82e389e3995a,bf5cb443-292a-48f3-b713-b4c7ebb7344d,2024-12-02 06:26:00,2024-12-02 09:12:12
2,9fb932d4-f039-4722-96ff-82e389e3995a,14b062ae-88c8-4ad1-aee1-f220fd547512,2024-12-02 14:12:00,2024-12-02 19:07:47
3,5ccedc77-3429-479c-acda-4ccb01f35efe,0d2329d9-fa09-4a46-b673-89669b02a56d,2024-12-02 14:11:00,2024-12-02 17:21:47
4,9fb932d4-f039-4722-96ff-82e389e3995a,f3d2273a-3740-4227-a10f-4d0c5b86c0ef,2024-12-02 22:19:00,2024-12-03 01:33:14



<class 'pandas.DataFrame'>
RangeIndex: 2100 entries, 0 to 2099
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   employee_id  2100 non-null   str           
 1   machine_id   2100 non-null   str           
 2   time_in      2100 non-null   datetime64[us]
 3   time_out     2100 non-null   datetime64[us]
dtypes: datetime64[us](2), str(2)
memory usage: 65.8 KB


---
## 3. Domain: `maintenance`

**Goal:** Parse timestamps, handle missing values, and correct records where the fix appears to precede the request.

This domain has three distinct data quality issues, each requiring a different strategy.

### Dirty Input

In [48]:
maintenance.head(8)

,request_timestamp,fix_timestamp,reason_code,requestor_ID
0,12/03/24 09:16:07,12/03/24 13:16:07,af11c1e7-7946-4a34-abd2-ed1c05f73dba,ad689cf8-8759-4153-b778-5728f2655b19
1,12/11/24 20:58:33,12/11/24 22:06,af11c1e7-7946-4a34-abd2-ed1c05f73dba,b35331ce-af2e-49dd-87e3-55b26210b784
2,12/19/24 16:50,12/19/24 14:05:00,f0fc795a-5a93-41f6-a058-8857c3881083,8ed5f036-44f7-48cd-8aeb-34f967124890
3,12/24/24 03:48:43,12/24/24 06:01:01,d33bf820-6770-4a78-8ba3-fb8041f089dc,ad689cf8-8759-4153-b778-5728f2655b19
4,01/02/25 22:19:00,01/03/25 02:19:00,618d3f43-0173-445f-9e0d-54254f4f81b0,9974d75b-3338-44fe-a179-0134676b1b69
5,01/06/25 10:23:54,01/06/25 14:07:00,7166a9f5-3620-4dbc-a3f3-760f0e2eb028,50fe3281-7812-4170-967a-34d0c643e653
6,03/21/25 02:50:41,03/21/25 06:06,efca6c3a-14c6-495e-ab45-1936d1d5c42f,8ed5f036-44f7-48cd-8aeb-34f967124890
7,04/01/25 09:55:50,04/01/25 13:55:50,af11c1e7-7946-4a34-abd2-ed1c05f73dba,b8e3c71f-6bf0-4d62-b310-57ca7d411fab


In [49]:
print('Missing values:')
print(maintenance.isna().sum())

print(f'\nRows with missing fix_timestamp:')
display(maintenance[maintenance['fix_timestamp'].isna()][['request_timestamp', 'fix_timestamp', 'requestor_ID']].head(5))

Missing values:
request_timestamp     0
fix_timestamp        71
reason_code           0
requestor_ID          0
dtype: int64

Rows with missing fix_timestamp:


,request_timestamp,fix_timestamp,requestor_ID
14,12/06/24 09:40:19,NaN,d283eb3a-5fbd-438e-89cf-158de6e96d45
30,04/17/25 19:17:49,NaN,baa1c6f1-404b-4eaf-962a-01dec28753f8
31,05/10/25 04:50:29,NaN,b8e3c71f-6bf0-4d62-b310-57ca7d411fab
33,05/15/25 05:26:51,NaN,ad689cf8-8759-4153-b778-5728f2655b19
34,05/21/25 19:56:37,NaN,9fb932d4-f039-4722-96ff-82e389e3995a


In [50]:
# Illustrate the two timestamp formats side-by-side
with_seconds    = maintenance[maintenance['request_timestamp'].str.len() > 14]['request_timestamp'].head(3)
without_seconds = maintenance[maintenance['request_timestamp'].str.len() == 14]['request_timestamp'].head(3)

print('With seconds    :', list(with_seconds))
print('Without seconds :', list(without_seconds))

With seconds    : ['12/03/24 09:16:07', '12/11/24 20:58:33', '12/24/24 03:48:43']
Without seconds : ['12/19/24 16:50', '08/16/25 02:34', '04/12/25 01:09']


### Cleaning — Step 1: Parse Inconsistent Timestamps

The format is `MM/DD/YY HH:MM:SS` when seconds are present, and `MM/DD/YY HH:MM` when they are not. `pd.to_datetime` with a single format string would fail on one of them, so we inspect the length to pick the right format.

In [51]:
def parse_maint_ts(ts_str):
    if pd.isna(ts_str):
        return pd.NaT
    ts_str = ts_str.strip()
    if len(ts_str) == 14:  # 'MM/DD/YY HH:MM' — no seconds
        return pd.to_datetime(ts_str, format='%m/%d/%y %H:%M')
    return pd.to_datetime(ts_str, format='%m/%d/%y %H:%M:%S')

maintenance['request_timestamp'] = maintenance['request_timestamp'].apply(parse_maint_ts) # type: ignore
maintenance['fix_timestamp']     = maintenance['fix_timestamp'].apply(parse_maint_ts) # type: ignore

print('Dtypes after parsing:')
print(maintenance[['request_timestamp', 'fix_timestamp']].dtypes)

Dtypes after parsing:
request_timestamp    datetime64[us]
fix_timestamp        datetime64[us]
dtype: object


In [52]:
# Spot-check a few rows with/without seconds
maintenance[['request_timestamp', 'fix_timestamp']].head(5)

,request_timestamp,fix_timestamp
0,2024-12-03 09:16:07,2024-12-03 13:16:07
1,2024-12-11 20:58:33,2024-12-11 22:06:00
2,2024-12-19 16:50:00,2024-12-19 14:05:00
3,2024-12-24 03:48:43,2024-12-24 06:01:01
4,2025-01-02 22:19:00,2025-01-03 02:19:00


### Cleaning — Step 2: Impute Missing `fix_timestamp`

13 rows have no `fix_timestamp` — the repair was logged but the completion time was not recorded. 

**Design decision:** Use `request_timestamp + 4 hours` as a reasonable proxy for typical repair turnaround time. We document this assumption so anyone reading the data knows these are imputed, not observed.

In [53]:
# Capture the null indexes so we can verify the fix
null_fix_idx = maintenance[maintenance['fix_timestamp'].isna()].index
print(f'Rows with missing fix_timestamp: {len(null_fix_idx)}')

# Impute: request_timestamp + 4 hours
maintenance.loc[null_fix_idx, 'fix_timestamp'] = (
    maintenance.loc[null_fix_idx, 'request_timestamp'] + pd.to_timedelta('4 hours')
)

assert maintenance['fix_timestamp'].isna().sum() == 0, 'Still have nulls!'
print('No missing fix_timestamps remaining ✓')

Rows with missing fix_timestamp: 71
No missing fix_timestamps remaining ✓


In [54]:
# Show the imputed rows — fix_timestamp is exactly 4 hours after request
maintenance.loc[null_fix_idx, ['request_timestamp', 'fix_timestamp']].head(5)

,request_timestamp,fix_timestamp
14,2024-12-06 09:40:19,2024-12-06 13:40:19
30,2025-04-17 19:17:49,2025-04-17 23:17:49
31,2025-05-10 04:50:29,2025-05-10 08:50:29
33,2025-05-15 05:26:51,2025-05-15 09:26:51
34,2025-05-21 19:56:37,2025-05-21 23:56:37


### Cleaning — Step 3: Correct Time-Travel Records

16 rows have `fix_timestamp < request_timestamp` — the machine was supposedly fixed *before* the repair was requested. This is physically impossible; it indicates a data entry error (likely a date transposition or a copy-paste mistake).

**Design decision:** Replace the erroneous `fix_timestamp` with `request_timestamp + 30 minutes`. This is a conservative lower bound — 30 minutes is a plausible minimum repair time.

In [55]:
early_fix_mask = maintenance['fix_timestamp'] < maintenance['request_timestamp']
print(f'Time-travel records: {early_fix_mask.sum()}')

maintenance.loc[early_fix_mask, ['request_timestamp', 'fix_timestamp']]

Time-travel records: 70


,request_timestamp,fix_timestamp
2,2024-12-19 16:50:00,2024-12-19 14:05:00
8,2025-04-21 09:40:34,2025-04-21 09:38:38
10,2025-07-26 02:21:32,2025-07-26 02:14:32
19,2025-02-01 03:56:53,2025-02-01 01:29:32
20,2025-02-05 19:26:41,2025-02-05 18:41:47
...,...,...
650,2025-09-08 19:16:41,2025-09-08 17:24:42
657,2025-09-23 09:12:51,2025-09-23 06:24:00
662,2025-10-01 20:21:07,2025-10-01 18:44:00
683,2025-11-13 10:49:49,2025-11-13 10:38:49


In [56]:
maintenance.loc[early_fix_mask, 'fix_timestamp'] = (
    maintenance.loc[early_fix_mask, 'request_timestamp'] + pd.to_timedelta('30 minutes')
)

remaining = (maintenance['fix_timestamp'] < maintenance['request_timestamp']).sum()
assert remaining == 0, 'Still have time-travel records!'
print('All time-travel records corrected ✓')

All time-travel records corrected ✓


### Clean Output ✓

In [57]:
# Show the corrected time-travel rows side-by-side
print('Corrected rows (fix is now 30 min after request):')
display(maintenance.loc[early_fix_mask, ['request_timestamp', 'fix_timestamp']].head(5))

print('\nOverall info:')
maintenance.info()

Corrected rows (fix is now 30 min after request):


,request_timestamp,fix_timestamp
2,2024-12-19 16:50:00,2024-12-19 17:20:00
8,2025-04-21 09:40:34,2025-04-21 10:10:34
10,2025-07-26 02:21:32,2025-07-26 02:51:32
19,2025-02-01 03:56:53,2025-02-01 04:26:53
20,2025-02-05 19:26:41,2025-02-05 19:56:41



Overall info:
<class 'pandas.DataFrame'>
RangeIndex: 707 entries, 0 to 706
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   request_timestamp  707 non-null    datetime64[us]
 1   fix_timestamp      707 non-null    datetime64[us]
 2   reason_code        707 non-null    str           
 3   requestor_ID       707 non-null    str           
dtypes: datetime64[us](2), str(2)
memory usage: 22.2 KB


---
## 4. Domain: `shift_schedule`

**Goal:** Standardize the `Title` column and parse `clock_in` / `clock_out` into full datetime objects.

This is the most complex domain. The clock time parsing requires knowledge of *which shift* a row belongs to in order to resolve the AM/PM ambiguity when the indicator is missing.

### Dirty Input

In [58]:
print('Title value counts (all mean the same two roles):')
print(shift_schedule['Title'].value_counts())

Title value counts (all mean the same two roles):
Title
line       943
Manager    581
Line       318
LINE       306
manager    170
Name: count, dtype: int64


In [59]:
print('clock_in value counts (note entries with and without AM/PM):')
print(shift_schedule['clock_in'].value_counts().head(12))

clock_in value counts (note entries with and without AM/PM):
clock_in
2:00 PM     345
10:00 PM    315
6:00 AM     309
2:05 PM     118
10:15 PM    117
10:05 PM    116
6:15 AM     115
2:10 PM     112
2:15 PM     110
6:05 AM     106
10:10 PM    105
6:10 AM     103
Name: count, dtype: int64


In [60]:
# A sample that shows all three shifts and both time formats
shift_schedule[['employee_ID', 'shift', 'date', 'clock_in', 'clock_out', 'Title']].head(10)

,employee_ID,shift,date,clock_in,clock_out,Title
0,9974d75b-3338-44fe-a179-0134676b1b69,1,12/02/2024,6:15 AM,2:00,Manager
1,5ccedc77-3429-479c-acda-4ccb01f35efe,1,12/02/2024,6:00,2:00 PM,LINE
2,9fb932d4-f039-4722-96ff-82e389e3995a,1,12/02/2024,6:15 AM,2:15 PM,LINE
3,9974d75b-3338-44fe-a179-0134676b1b69,2,12/02/2024,2:00 PM,10:15 PM,Manager
4,9fb932d4-f039-4722-96ff-82e389e3995a,2,12/02/2024,2:00 PM,10:00 PM,Line
5,5ccedc77-3429-479c-acda-4ccb01f35efe,2,12/02/2024,2:05 PM,10:00 PM,LINE
6,221c4e00-3f99-41ee-baf2-7f802dc5fd3d,3,12/02/2024,10:10 PM,6:00,Manager
7,9fb932d4-f039-4722-96ff-82e389e3995a,3,12/02/2024,10:05 PM,6:00 AM,line
8,cae8c077-3779-45b3-96a1-da2c9cfbba43,3,12/02/2024,10:00 PM,6:15 AM,line
9,9974d75b-3338-44fe-a179-0134676b1b69,1,12/03/2024,6:10 AM,2:15 PM,Manager


### Cleaning — Step 1: Normalize `Title`

Strip whitespace, lowercase, then map to canonical role names. We rename `line` → `Operator` to use a more descriptive term than the original shorthand.

In [61]:
TITLE_MAP = {'line': 'Operator', 'manager': 'Manager'}

shift_schedule['title'] = (
    shift_schedule['Title']
    .str.strip()
    .str.lower()
    .map(TITLE_MAP)
)

shift_schedule.drop(columns=['Title'], inplace=True)

print('After normalization:')
print(shift_schedule['title'].value_counts())

After normalization:
title
Operator    1567
Manager      751
Name: count, dtype: int64


### Cleaning — Step 2: Parse Clock Times

The shift definitions give us the context we need to resolve ambiguous times:

| Shift | Start range | End range |
|-------|-------------|----------|
| 1 (Day)   | ~6 AM  | ~2 PM  |
| 2 (Swing) | ~2 PM  | ~10 PM |
| 3 (Night) | ~10 PM | ~6 AM  |

When the AM/PM indicator is missing (`6:15` instead of `6:15 AM`), we compare the bare hour against the shift's expected hour range to determine whether to add 12.

In [62]:
# Canonical shift start/end hours — used to resolve ambiguous times
SHIFT_HOURS = {
    1: {'clock_in': 6,  'clock_out': 14},
    2: {'clock_in': 14, 'clock_out': 22},
    3: {'clock_in': 22, 'clock_out': 6},
}

def adjust_hour(hour, shift, col):
    """Add 12 to a bare hour if the shift says it should be in the PM/night range."""
    expected = SHIFT_HOURS[shift][col]
    if hour <= 12 and expected > 12:
        return hour + 12
    return hour

def parse_clock_time(row, col):
    clock_str = row[col]
    date_dt   = pd.to_datetime(row['date'], format='%m/%d/%Y').date()

    if ' ' in clock_str:                         # '6:15 AM' / '10:00 PM'
        t = datetime.strptime(clock_str, '%I:%M %p').time()
    else:                                         # '6:15' / '10:00' — no AM/PM
        h, m = map(int, clock_str.split(':'))
        h = adjust_hour(h, row['shift'], col)
        t = time(h, m)

    return datetime.combine(date_dt, t)

shift_schedule['clock_in_dt']  = shift_schedule.apply(lambda r: parse_clock_time(r, 'clock_in'),  axis=1)
shift_schedule['clock_out_dt'] = shift_schedule.apply(lambda r: parse_clock_time(r, 'clock_out'), axis=1)

shift_schedule.drop(columns=['clock_in', 'clock_out', 'date'], inplace=True)
shift_schedule.rename(columns={'employee_ID': 'employee_id'}, inplace=True)

### Clean Output ✓

In [63]:
print('title counts:')
print(shift_schedule['title'].value_counts())
print()

# One row from each shift to verify times look right
sample = shift_schedule.groupby('shift').first().reset_index()
display(sample[['shift', 'employee_id', 'clock_in_dt', 'clock_out_dt', 'title']])
print()
shift_schedule.info()

title counts:
title
Operator    1567
Manager      751
Name: count, dtype: int64



,shift,employee_id,clock_in_dt,clock_out_dt,title
0,1,9974d75b-3338-44fe-a179-0134676b1b69,2024-12-02 06:15:00,2024-12-02 14:00:00,Manager
1,2,9974d75b-3338-44fe-a179-0134676b1b69,2024-12-02 14:00:00,2024-12-02 22:15:00,Manager
2,3,221c4e00-3f99-41ee-baf2-7f802dc5fd3d,2024-12-02 22:10:00,2024-12-02 06:00:00,Manager



<class 'pandas.DataFrame'>
RangeIndex: 2318 entries, 0 to 2317
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   employee_id   2318 non-null   str           
 1   shift         2318 non-null   int64         
 2   title         2318 non-null   str           
 3   clock_in_dt   2318 non-null   datetime64[us]
 4   clock_out_dt  2318 non-null   datetime64[us]
dtypes: datetime64[us](2), int64(1), str(2)
memory usage: 90.7 KB


---
## 5. Building the index

Now that each domain is clean we integrate them into a single unified table: **`machine_usage`**.

The key challenge is that the source tables don't share a direct foreign key. We use **`pd.merge_asof`** — a time-ordered nearest-match join — to correlate records across tables.

The full schema of the index (Step 6 adds precomputed analytics fields on top of this):

| Column | Source | Description |
|--------|--------|-------------|
| `id` | computed | Composite key: `{machine_id}_{start}_{end}` |
| `machine_id` | machine_log / maintenance→machine_log | Machine UUID |
| `personnel` | machine_log.employee_id / maintenance.requestor_id | Person UUID |
| `title` | shift_schedule.title / hardcoded | `Operator`, `Manager`, or `Maintenance` |
| `type` | hardcoded | `Regular` or `Maintenance` |
| `start_timestamp` | machine_log.time_in / maintenance.request_timestamp | |
| `end_timestamp` | machine_log.time_out / maintenance.fix_timestamp | |
| `shift_name` | derived from `start_timestamp` hour | `Day`, `Swing`, or `Night` |
| `reason_code` | maintenance.reason_code | Maintenance records only; `null` for Regular |

### Step 1: Add `title` to Machine Log Records

For each machine log entry, find the shift schedule record for the same employee whose `clock_in_dt` is the latest one **on or before** the machine log `time_in`. This gives us the title (Operator / Manager) in effect at that time.

In [64]:
shift_sorted = shift_schedule[['employee_id', 'clock_in_dt', 'title']].sort_values('clock_in_dt')

machine_log_with_title = pd.merge_asof(
    machine_log.sort_values('time_in'),
    shift_sorted,
    left_on='time_in',
    right_on='clock_in_dt',
    by='employee_id',
    direction='backward',
)

print(f'Rows: {len(machine_log_with_title)}')
print(f'Title nulls before fix: {machine_log_with_title["title"].isna().sum()}')

# The backward merge only matches if a shift entry exists *before* time_in.
# The very first sessions of the day arrive before clock_in is logged, so no
# prior entry is found. Use each employee's title from shift_schedule as the
# fallback — this correctly preserves Manager vs Operator rather than
# blindly defaulting everything to Operator.
title_by_employee = shift_schedule.groupby('employee_id')['title'].first()

null_mask = machine_log_with_title['title'].isna()
machine_log_with_title.loc[null_mask, 'title'] = (
    machine_log_with_title.loc[null_mask, 'employee_id']
    .map(title_by_employee)
)

print(f'Title nulls after fix:  {machine_log_with_title["title"].isna().sum()}')
display(machine_log_with_title[['employee_id', 'machine_id', 'time_in', 'time_out', 'title']].head(5))

Rows: 2100
Title nulls before fix: 1
Title nulls after fix:  0


,employee_id,machine_id,time_in,time_out,title
0,9fb932d4-f039-4722-96ff-82e389e3995a,bf5cb443-292a-48f3-b713-b4c7ebb7344d,2024-12-02 06:26:00,2024-12-02 09:12:12,Operator
1,5ccedc77-3429-479c-acda-4ccb01f35efe,32b55d35-e676-46f1-84c6-9290764b5018,2024-12-02 06:28:00,2024-12-02 12:23:22,Operator
2,5ccedc77-3429-479c-acda-4ccb01f35efe,0d2329d9-fa09-4a46-b673-89669b02a56d,2024-12-02 14:11:00,2024-12-02 17:21:47,Operator
3,9fb932d4-f039-4722-96ff-82e389e3995a,14b062ae-88c8-4ad1-aee1-f220fd547512,2024-12-02 14:12:00,2024-12-02 19:07:47,Operator
4,cae8c077-3779-45b3-96a1-da2c9cfbba43,bf5cb443-292a-48f3-b713-b4c7ebb7344d,2024-12-02 22:16:00,2024-12-03 03:36:20,Operator


### Step 2: Add `machine_id` to Maintenance Records

Maintenance requests don't record which machine broke down. We infer it by finding the machine log entry where:
- The same person (`requestor_ID == employee_id`) was last seen
- Their session ended (`time_out`) **at or before** the request timestamp

The assumption: the machine you were just working on is the one you're reporting.

**Fallback:** Managers file reports but don't appear in `machine_log`. For those rows we fall back to the globally most recent machine log entry before the request.

In [65]:
# Rename maintenance columns for the merge
maintenance.rename(columns={'requestor_ID': 'requestor_id'}, inplace=True)

machine_log_sorted = machine_log.sort_values('time_out')
maintenance_sorted = maintenance.sort_values('request_timestamp')

maintenance = pd.merge_asof(
    maintenance_sorted,
    machine_log_sorted[['employee_id', 'machine_id', 'time_out']],
    left_on='request_timestamp',
    right_on='time_out',
    left_by='requestor_id',
    right_by='employee_id',
    direction='backward',
)

unmatched = maintenance['machine_id'].isna().sum()
print(f'Matched on first pass: {len(maintenance) - unmatched}/{len(maintenance)}')
print(f'Unmatched (likely managers): {unmatched}')

Matched on first pass: 634/707
Unmatched (likely managers): 73


In [66]:
# Global fallback: most recent machine session before the request, regardless of who was on it
def find_global_match(request_dt):
    candidates = machine_log_sorted[machine_log_sorted['time_out'] <= request_dt]
    if candidates.empty:
        return pd.Series([None, None, None], index=['machine_id', 'employee_id', 'time_out'])
    return candidates.iloc[-1][['machine_id', 'employee_id', 'time_out']]

unmatched_mask = maintenance['machine_id'].isna()

maintenance.loc[unmatched_mask, ['machine_id', 'employee_id', 'time_out']] = (
    maintenance.loc[unmatched_mask, 'request_timestamp']
    .apply(find_global_match)
    .values
)

assert maintenance['machine_id'].isna().sum() == 0
print('All maintenance records have a machine_id ✓')

# Show a sample of the manager-submitted rows that got the fallback
display(maintenance.loc[unmatched_mask, ['requestor_id', 'request_timestamp', 'machine_id']].head(5))

All maintenance records have a machine_id ✓


,requestor_id,request_timestamp,machine_id
18,221c4e00-3f99-41ee-baf2-7f802dc5fd3d,2024-12-13 01:06:00,0d2329d9-fa09-4a46-b673-89669b02a56d
29,baa1c6f1-404b-4eaf-962a-01dec28753f8,2024-12-19 11:38:00,1c1b5b32-097c-480c-ad0f-d6d9a90965f5
43,baa1c6f1-404b-4eaf-962a-01dec28753f8,2024-12-30 08:50:45,1c1b5b32-097c-480c-ad0f-d6d9a90965f5
52,9974d75b-3338-44fe-a179-0134676b1b69,2025-01-02 22:19:00,14b062ae-88c8-4ad1-aee1-f220fd547512
54,9974d75b-3338-44fe-a179-0134676b1b69,2025-01-03 02:48:11,0d2329d9-fa09-4a46-b673-89669b02a56d


### Step 3: Build the Unified `machine_usage` Table

Project each source into a common schema, then concatenate.

In [67]:
def make_id(machine_id, start, end):
    return f'{machine_id}_{start.isoformat()}_{end.isoformat()}'

# Regular records
regular = machine_log_with_title[['machine_id', 'employee_id', 'title', 'time_in', 'time_out']].copy()
regular['id']   = regular.apply(lambda r: make_id(r.machine_id, r.time_in, r.time_out), axis=1)
regular['type'] = 'Regular'
regular.rename(columns={'employee_id': 'personnel', 'time_in': 'start_timestamp', 'time_out': 'end_timestamp'}, inplace=True)

# Maintenance records
maint = maintenance[['machine_id', 'requestor_id', 'request_timestamp', 'fix_timestamp']].copy()
maint['id']    = maint.apply(lambda r: make_id(r.machine_id, r.request_timestamp, r.fix_timestamp), axis=1)
maint['title'] = 'Maintenance'
maint['type']  = 'Maintenance'
maint.rename(columns={'requestor_id': 'personnel', 'request_timestamp': 'start_timestamp', 'fix_timestamp': 'end_timestamp'}, inplace=True)

machine_usage = pd.concat(
    [regular[['id', 'machine_id', 'personnel', 'title', 'type', 'start_timestamp', 'end_timestamp']],
     maint[  ['id', 'machine_id', 'personnel', 'title', 'type', 'start_timestamp', 'end_timestamp']]],
    ignore_index=True,
)

print(f'Total rows: {len(machine_usage)}')
print()
print('Type breakdown:')
print(machine_usage['type'].value_counts())
print()
print(f'Date range: {machine_usage["start_timestamp"].min()}  →  {machine_usage["end_timestamp"].max()}')

Total rows: 2807

Type breakdown:
type
Regular        2100
Maintenance     707
Name: count, dtype: int64

Date range: 2024-12-02 06:26:00  →  2026-01-01 02:30:23


In [68]:
machine_usage.head(8)

,id,machine_id,personnel,title,type,start_timestamp,end_timestamp
0,bf5cb443-292a-48f3-b713-b4c7ebb7344d_2024-12-0...,bf5cb443-292a-48f3-b713-b4c7ebb7344d,9fb932d4-f039-4722-96ff-82e389e3995a,Operator,Regular,2024-12-02 06:26:00,2024-12-02 09:12:12
1,32b55d35-e676-46f1-84c6-9290764b5018_2024-12-0...,32b55d35-e676-46f1-84c6-9290764b5018,5ccedc77-3429-479c-acda-4ccb01f35efe,Operator,Regular,2024-12-02 06:28:00,2024-12-02 12:23:22
2,0d2329d9-fa09-4a46-b673-89669b02a56d_2024-12-0...,0d2329d9-fa09-4a46-b673-89669b02a56d,5ccedc77-3429-479c-acda-4ccb01f35efe,Operator,Regular,2024-12-02 14:11:00,2024-12-02 17:21:47
3,14b062ae-88c8-4ad1-aee1-f220fd547512_2024-12-0...,14b062ae-88c8-4ad1-aee1-f220fd547512,9fb932d4-f039-4722-96ff-82e389e3995a,Operator,Regular,2024-12-02 14:12:00,2024-12-02 19:07:47
4,bf5cb443-292a-48f3-b713-b4c7ebb7344d_2024-12-0...,bf5cb443-292a-48f3-b713-b4c7ebb7344d,cae8c077-3779-45b3-96a1-da2c9cfbba43,Operator,Regular,2024-12-02 22:16:00,2024-12-03 03:36:20
5,f3d2273a-3740-4227-a10f-4d0c5b86c0ef_2024-12-0...,f3d2273a-3740-4227-a10f-4d0c5b86c0ef,9fb932d4-f039-4722-96ff-82e389e3995a,Operator,Regular,2024-12-02 22:19:00,2024-12-03 01:33:14
6,32b55d35-e676-46f1-84c6-9290764b5018_2024-12-0...,32b55d35-e676-46f1-84c6-9290764b5018,9fb932d4-f039-4722-96ff-82e389e3995a,Operator,Regular,2024-12-03 01:49:14,2024-12-03 05:16:19
7,80d16aaf-f1a4-4fe5-ad78-dc4bfb9e8dda_2024-12-0...,80d16aaf-f1a4-4fe5-ad78-dc4bfb9e8dda,cae8c077-3779-45b3-96a1-da2c9cfbba43,Operator,Regular,2024-12-03 06:10:00,2024-12-03 08:23:01


### Step 4: Add Derived Fields

Two fields can be added directly from data we already have — no aggregation or window logic needed:

- **`shift_name`** — derived from the hour of `start_timestamp`: `Day` (6–14), `Swing` (14–22), `Night` (otherwise)
- **`reason_code`** — for Maintenance records only; looked up by `(personnel, start_timestamp)` from the maintenance table

In [69]:
# --- shift_name ---
def infer_shift_name(hour):
    if 6 <= hour < 14: return 'Day'
    if 14 <= hour < 22: return 'Swing'
    return 'Night'

machine_usage['shift_name'] = machine_usage['start_timestamp'].dt.hour.map(infer_shift_name)

# --- reason_code (Maintenance records only) ---
# maintenance still uses its original column name request_timestamp here;
# the rename to start_timestamp only exists on the maint slice inside step3-concat.
reason_lookup = dict(zip(
    zip(maintenance['requestor_id'], maintenance['request_timestamp']),
    maintenance['reason_code'],
))

machine_usage['reason_code'] = machine_usage.apply(
    lambda r: reason_lookup.get((r['personnel'], r['start_timestamp']))
              if r['type'] == 'Maintenance' else None,
    axis=1,
)

print('shift_name counts:', machine_usage['shift_name'].value_counts().to_dict())
print('reason_code non-null:', machine_usage['reason_code'].notna().sum(), '(should equal 198)')

# Show a mix of both types so reason_code population is visible
cols = ['machine_id', 'personnel', 'title', 'type', 'start_timestamp', 'shift_name', 'reason_code']
sample = pd.concat([
    machine_usage[machine_usage['type'] == 'Regular'].head(3),
    machine_usage[machine_usage['type'] == 'Maintenance'].head(3),
]).reset_index(drop=True)
display(sample[cols])

shift_name counts: {'Swing': 974, 'Night': 922, 'Day': 911}
reason_code non-null: 707 (should equal 198)


,machine_id,personnel,title,type,start_timestamp,shift_name,reason_code
0,bf5cb443-292a-48f3-b713-b4c7ebb7344d,9fb932d4-f039-4722-96ff-82e389e3995a,Operator,Regular,2024-12-02 06:26:00,Day,NaN
1,32b55d35-e676-46f1-84c6-9290764b5018,5ccedc77-3429-479c-acda-4ccb01f35efe,Operator,Regular,2024-12-02 06:28:00,Day,NaN
2,0d2329d9-fa09-4a46-b673-89669b02a56d,5ccedc77-3429-479c-acda-4ccb01f35efe,Operator,Regular,2024-12-02 14:11:00,Swing,NaN
3,bf5cb443-292a-48f3-b713-b4c7ebb7344d,cae8c077-3779-45b3-96a1-da2c9cfbba43,Maintenance,Maintenance,2024-12-03 03:42:00,Night,618d3f43-0173-445f-9e0d-54254f4f81b0
4,80d16aaf-f1a4-4fe5-ad78-dc4bfb9e8dda,cae8c077-3779-45b3-96a1-da2c9cfbba43,Maintenance,Maintenance,2024-12-03 08:54:01,Day,0e4a1175-ec15-4269-b9e4-8824b1eb72c8
5,14b062ae-88c8-4ad1-aee1-f220fd547512,ad689cf8-8759-4153-b778-5728f2655b19,Maintenance,Maintenance,2024-12-03 09:16:07,Day,af11c1e7-7946-4a34-abd2-ed1c05f73dba


---
## 6. Load into OpenSearch

### Why precompute analytics fields at index time?

OpenSearch is a search engine, not a relational database. It does not support window functions, CTEs, or temporal JOINs. Fields we want to filter or aggregate on must be present in each document at index time.

This is the core tradeoff of the bronze layer: **compute at write time so reads are trivial.**

### Connect to OpenSearch

Make sure Docker is running: `docker compose -f opensearch/docker-compose.yml up -d`

In [70]:
client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    use_ssl=False,
)

_os_ready = False
try:
    info = client.info()
    print(f'Connected to OpenSearch {info["version"]["number"]} ✓')
    _os_ready = True
except Exception:
    print('❌ OpenSearch is not reachable on localhost:9200.')
    print('   Start it with: docker compose -f opensearch/docker-compose.yml up -d')
    print('   Then re-run this cell before continuing.')

Connected to OpenSearch 2.11.0 ✓


### Define the Index Schema as a Document Class

Each field in `machine_usage` maps to a typed DSL field. The inner `Index` class holds the index name and settings — `MachineUsage.init(using=client)` reads these to create the index with the correct mapping.

In [71]:
class MachineUsage(Document):
    machine_id                  = Keyword()
    personnel                   = Keyword()
    title                       = Keyword()
    type                        = Keyword()
    start_timestamp             = Date()
    end_timestamp               = Date()
    duration_hours              = Float()
    shift_name                  = Keyword()
    shift_active_hours          = Float()
    employee_idle_hours_from_prev = Float()
    machine_gap_hours_from_prev = Float()
    reason_code                 = Keyword()

    class Index:
        name = 'machine_usage'
        settings = {
            'number_of_shards': 1,
            'number_of_replicas': 0,
        }

print('MachineUsage document class defined ✓')

MachineUsage document class defined ✓


In [72]:
assert _os_ready, 'OpenSearch is not connected — re-run the cell above first.'

# Delete first so re-running this notebook is idempotent
client.indices.delete(index=MachineUsage.Index.name)

# init() reads the Document class definition and creates the index with
# the correct mapping and settings — no hand-written dict required.
MachineUsage.init(using=client)
print(f'Index "{MachineUsage.Index.name}" created ✓')

Index "machine_usage" created ✓


### Enrichment: Precomputing Aggregate & Window Fields

The remaining fields require looking *across* multiple records — aggregation or LAG-style window logic that OpenSearch cannot perform at query time. We compute them here and store them in each document.

| Field | Description |
|-------|-------------|
| `duration_hours` | End minus start in fractional hours |
| `employee_idle_hours_from_prev` | Idle hours between the previous session's end and this session's start, for the same person within the same shift |
| `machine_gap_hours_from_prev` | Hours since the same machine's previous session (any type) |
| `shift_active_hours` | Total Regular hours for this person across all sessions in this shift |

In [73]:
# --- duration_hours ---
machine_usage['duration_hours'] = (
    (machine_usage['end_timestamp'] - machine_usage['start_timestamp'])
    .dt.total_seconds() / 3600
).round(2)

print('duration_hours sample:', machine_usage['duration_hours'].head(3).tolist())

duration_hours sample: [2.77, 5.92, 3.18]


In [74]:
# --- employee_idle_hours_from_prev ---
# For Regular records: idle time between the previous session's end and this session's start,
# for the same person within the same shift.
# Night shift sessions that start 00:00–05:59 belong to the *previous* day's canonical shift.

def canonical_shift_key(ts):
    h = ts.hour
    num = 1 if 6 <= h < 14 else (2 if 14 <= h < 22 else 3)
    d = ts.date()
    if num == 3 and h < 6:
        d -= timedelta(days=1)
    return (num, d)

machine_usage['_shift_key'] = machine_usage['start_timestamp'].apply(canonical_shift_key)

reg_mask = machine_usage['type'] == 'Regular'
reg = machine_usage[reg_mask].sort_values(['personnel', '_shift_key', 'start_timestamp'])
prev_end = reg.groupby(['personnel', '_shift_key'])['end_timestamp'].shift(1)
machine_usage.loc[reg.index, 'employee_idle_hours_from_prev'] = (
    (reg['start_timestamp'] - prev_end).dt.total_seconds() / 3600
).round(2)

# --- machine_gap_hours_from_prev ---
# For all records: gap from previous usage of the same machine.
all_s = machine_usage.sort_values(['machine_id', 'start_timestamp'])
prev_machine_end = all_s.groupby('machine_id')['end_timestamp'].shift(1)
machine_usage.loc[all_s.index, 'machine_gap_hours_from_prev'] = (
    (all_s['start_timestamp'] - prev_machine_end).dt.total_seconds() / 3600
).round(2)

gapped = machine_usage['employee_idle_hours_from_prev'].notna().sum()
print(f'Records with employee_idle_hours_from_prev: {gapped}')

Records with employee_idle_hours_from_prev: 533


In [75]:
# --- shift_active_hours ---
# Total active Regular hours per (person, shift).

reg_df = machine_usage[reg_mask].copy()
reg_df['_dur'] = (reg_df['end_timestamp'] - reg_df['start_timestamp']).dt.total_seconds() / 3600

shift_totals = reg_df.groupby(['personnel', '_shift_key'])['_dur'].transform('sum').round(2)
machine_usage.loc[reg_mask, 'shift_active_hours'] = shift_totals.values

# Drop the helper column used by both gap cells
machine_usage.drop(columns=['_shift_key'], inplace=True)

print('Enrichment complete. Final columns:')
print(list(machine_usage.columns))

Enrichment complete. Final columns:
['id', 'machine_id', 'personnel', 'title', 'type', 'start_timestamp', 'end_timestamp', 'shift_name', 'reason_code', 'duration_hours', 'employee_idle_hours_from_prev', 'machine_gap_hours_from_prev', 'shift_active_hours']


In [76]:
cols = ['id', 'type', 'duration_hours', 'shift_name',
        'employee_idle_hours_from_prev', 'shift_active_hours',
        'machine_gap_hours_from_prev']

reg_with_gap = machine_usage[machine_usage['employee_idle_hours_from_prev'].notna()].head(3)
reg_first    = machine_usage[machine_usage['type'] == 'Regular'].iloc[4:6]
maint_rows   = machine_usage[machine_usage['type'] == 'Maintenance'].head(2)

sample = pd.concat([reg_first, reg_with_gap, maint_rows]).drop_duplicates().reset_index(drop=True)
display(sample[cols])

,id,type,duration_hours,shift_name,employee_idle_hours_from_prev,shift_active_hours,machine_gap_hours_from_prev
0,bf5cb443-292a-48f3-b713-b4c7ebb7344d_2024-12-0...,Regular,5.34,Night,NaN,5.34,13.06
1,f3d2273a-3740-4227-a10f-4d0c5b86c0ef_2024-12-0...,Regular,3.24,Night,NaN,6.69,NaN
2,32b55d35-e676-46f1-84c6-9290764b5018_2024-12-0...,Regular,3.45,Night,0.27,6.69,13.43
3,0d2329d9-fa09-4a46-b673-89669b02a56d_2024-12-0...,Regular,2.42,Day,0.65,4.64,15.67
4,80d16aaf-f1a4-4fe5-ad78-dc4bfb9e8dda_2024-12-0...,Regular,3.54,Day,0.40,5.94,8.24
5,bf5cb443-292a-48f3-b713-b4c7ebb7344d_2024-12-0...,Maintenance,3.09,Night,NaN,NaN,0.09
6,80d16aaf-f1a4-4fe5-ad78-dc4bfb9e8dda_2024-12-0...,Maintenance,1.55,Day,NaN,NaN,0.52


### Bulk Index

In [77]:
def _null(val):
    """Convert pandas NaN to None — OpenSearch rejects float('nan') as invalid JSON."""
    try:
        return None if math.isnan(val) else val
    except TypeError:
        return val

def row_to_doc(row):
    return MachineUsage(
        meta={'id': row['id']},
        machine_id                    = row['machine_id'],
        personnel                     = row['personnel'],
        title                         = row['title'],
        type                          = row['type'],
        start_timestamp               = row['start_timestamp'].isoformat(),
        end_timestamp                 = row['end_timestamp'].isoformat(),
        duration_hours                = row['duration_hours'],
        shift_name                    = row['shift_name'],
        shift_active_hours            = _null(row.get('shift_active_hours')),
        employee_idle_hours_from_prev = _null(row.get('employee_idle_hours_from_prev')),
        machine_gap_hours_from_prev   = _null(row.get('machine_gap_hours_from_prev')),
        reason_code                   = _null(row.get('reason_code')),
    )

actions = (row_to_doc(row).to_dict(include_meta=True) for _, row in machine_usage.iterrows())
success, errors = helpers.bulk(client, actions)

client.indices.refresh(index=MachineUsage.Index.name)

if errors:
    print(f'WARNING: {len(errors)} errors during indexing')
else:
    doc_count = client.count(index=MachineUsage.Index.name)['count']
    print(f'Indexed {success} documents. Index count: {doc_count} ✓')

Indexed 2807 documents. Index count: 2807 ✓


---
## 7. Exploring the index with the Search DSL

We query the index using `opensearch-dsl`'s `Search` class. The pattern:

```python
s = Search(using=client, index='machine_usage')
s = s.filter('term', field=value)   # narrow the result set
s = s.sort('field')                 # order results
s = s[:10]                          # size (slice notation)
resp = s.execute()

for hit in resp:                    # iterate hits directly
    print(hit.field_name)           # access fields as attributes
```

Aggregations are attached with `s.aggs.bucket(...)` and read from `resp.aggregations`.

### Query 1: Sample Documents (match_all)

The simplest possible query — returns any 5 documents. Good for a sanity check.

In [78]:
s = Search(using=client, index=MachineUsage.Index.name)[:5]
resp = s.execute()

print(f'Total documents in index: {resp.hits.total.value}')
print()
for hit in resp:
    print(f"{hit.type:<15} {hit.shift_name:<7} {hit.start_timestamp[:19]}  {hit.duration_hours:.2f}h  {hit.machine_id[:8]}...")

Total documents in index: 2807

Regular         Day     2024-12-02T06:26:00  2.77h  bf5cb443...
Regular         Day     2024-12-02T06:28:00  5.92h  32b55d35...
Regular         Swing   2024-12-02T14:11:00  3.18h  0d2329d9...
Regular         Swing   2024-12-02T14:12:00  4.93h  14b062ae...
Regular         Night   2024-12-02T22:16:00  5.34h  bf5cb443...


### Query 2: Maintenance Records Only

`.filter('term', ...)` performs an exact-match on a `keyword` field. Chaining multiple `.filter()` calls applies them as `bool / filter` clauses — all must match, and results are not scored (so OpenSearch can cache the filter automatically).

In [79]:
s = (
    Search(using=client, index=MachineUsage.Index.name)
    .filter('term', type='Maintenance')
    [:5]
)
resp = s.execute()

print(f'Maintenance records: {resp.hits.total.value}')
print()
for hit in resp:
    print(f"{hit.start_timestamp[:19]}  machine={hit.machine_id[:8]}...  duration={hit.duration_hours:.2f}h  reason={str(hit.reason_code)[:8]}...")

Maintenance records: 707

2024-12-03T03:42:00  machine=bf5cb443...  duration=3.09h  reason=618d3f43...
2024-12-03T08:54:01  machine=80d16aaf...  duration=1.55h  reason=0e4a1175...
2024-12-03T09:16:07  machine=14b062ae...  duration=4.00h  reason=af11c1e7...
2024-12-04T09:54:27  machine=ecc2c55e...  duration=1.19h  reason=618d3f43...
2024-12-04T13:20:15  machine=80d16aaf...  duration=2.73h  reason=0e4a1175...


### Query 3: All Sessions for a Specific Machine

Filter by `machine_id` to see everything that has happened to one machine — both Regular operator sessions and Maintenance events, in time order.

In [80]:
sample_machine = machine_usage['machine_id'].iloc[0]
print(f'Querying machine: {sample_machine}')

s = (
    Search(using=client, index=MachineUsage.Index.name)
    .filter('term', machine_id=sample_machine)
    .sort('start_timestamp')
    [:10]
)
resp = s.execute()

print(f'\nTotal sessions for this machine: {resp.hits.total.value}')
print()
print(f'{"Type":<15} {"Start":<22} {"Hours":>6}  Personnel')
print('-' * 70)
for hit in resp:
    print(f"{hit.type:<15} {hit.start_timestamp[:19]:<22} {hit.duration_hours:>6.2f}  {hit.personnel[:8]}...")

Querying machine: bf5cb443-292a-48f3-b713-b4c7ebb7344d

Total sessions for this machine: 354

Type            Start                   Hours  Personnel
----------------------------------------------------------------------
Regular         2024-12-02T06:26:00      2.77  9fb932d4...
Regular         2024-12-02T22:16:00      5.34  cae8c077...
Maintenance     2024-12-03T03:42:00      3.09  cae8c077...
Regular         2024-12-11T06:25:00      3.50  9fb932d4...
Regular         2024-12-12T06:28:00      4.12  50fe3281...
Maintenance     2024-12-12T10:47:00      4.00  50fe3281...
Regular         2024-12-12T22:17:00      5.34  9fb932d4...
Regular         2024-12-13T00:52:05      3.60  8ed5f036...
Maintenance     2024-12-13T04:13:00      4.00  9fb932d4...
Regular         2024-12-13T09:30:23      3.92  2dd301c8...


### Query 4: Top 3 Machines by Total Usage Hours

A `terms` aggregation groups documents by a `keyword` field (like `GROUP BY` in SQL). Nesting a `sum` metric aggregation inside it computes the total for each group.

In [81]:
s = Search(using=client, index=MachineUsage.Index.name)[:0]
s.aggs.bucket(
    'by_machine', 'terms',
    field='machine_id', size=3,
    order={'total_hours': 'desc'},
).metric('total_hours', 'sum', field='duration_hours') # type: ignore

resp = s.execute()

print('Top 3 machines by total usage hours:\n')
print(f'{"Machine ID":<40} {"Total Hours":>12}  {"Sessions":>9}')
print('-' * 65)
for bucket in resp.aggregations.by_machine.buckets:
    print(f"{bucket.key:<40} {bucket.total_hours.value:>12.1f}  {bucket.doc_count:>9}")

Top 3 machines by total usage hours:

Machine ID                                Total Hours   Sessions
-----------------------------------------------------------------
80d16aaf-f1a4-4fe5-ad78-dc4bfb9e8dda           1420.3        453
1c1b5b32-097c-480c-ad0f-d6d9a90965f5           1317.6        416
ecc2c55e-986d-4842-b114-3591cab5f7c4           1227.8        364


### Query 5: Maintenance Events in October 2025

Chain two `.filter()` calls — one for the exact type match, one for the date range. The DSL compiles these into a `bool / filter` clause automatically, so no manual nesting is needed.

In [82]:
s = (
    Search(using=client, index=MachineUsage.Index.name)
    .filter('term', type='Maintenance')
    .filter('range', start_timestamp={'gte': '2025-10-01', 'lt': '2025-11-01'})
    .sort('start_timestamp')
    [:20]
)
resp = s.execute()

print(f'Maintenance events in October 2025: {resp.hits.total.value}')
print()
print(f'{"Date":<22} {"Duration":>9}  {"Machine":<12}  Personnel')
print('-' * 70)
for hit in resp:
    print(f"{hit.start_timestamp[:19]:<22} {hit.duration_hours:>9.2f}h  {hit.machine_id[:8]+'...':<12}  {hit.personnel[:8]}...")

Maintenance events in October 2025: 63

Date                    Duration  Machine       Personnel
----------------------------------------------------------------------
2025-10-01T01:25:00         4.01h  f3d2273a...   b8e3c71f...
2025-10-01T01:55:00         4.00h  1c1b5b32...   9fb932d4...
2025-10-01T18:46:16         3.61h  ecc2c55e...   50fe3281...
2025-10-01T20:21:07         0.50h  1c1b5b32...   366c5acd...
2025-10-01T20:22:00         0.85h  80d16aaf...   b35331ce...
2025-10-01T21:59:02         1.60h  1c1b5b32...   50fe3281...
2025-10-02T01:01:48         3.99h  ecc2c55e...   366c5acd...
2025-10-02T04:22:09         1.70h  80d16aaf...   9fb932d4...
2025-10-02T11:04:37         3.12h  14b062ae...   ad689cf8...
2025-10-03T09:26:00         1.41h  ecc2c55e...   cae8c077...
2025-10-03T17:39:51         0.50h  bf5cb443...   ad689cf8...
2025-10-03T22:12:03         4.00h  bf5cb443...   cae8c077...
2025-10-06T09:05:42         4.00h  bf5cb443...   50fe3281...
2025-10-06T09:08:44         0.51h  80d

---
## Summary

In this lesson we went from three messy raw tables to a fully indexed, queryable index:

| Step | What we did |
|------|-------------|
| `machine_log` cleaning | Converted string timestamps → `datetime64` |
| `maintenance` cleaning | Parsed inconsistent formats, imputed 13 nulls, corrected 16 time-travel records |
| `shift_schedule` cleaning | Normalized mixed-case titles, resolved AM/PM ambiguity using shift context |
| Integration | Two `merge_asof` joins + global fallback → unified 814-row `machine_usage` table |
| Enrichment | Precomputed `duration_hours`, `shift_name`, gap fields, shift aggregates, `reason_code` |
| Load | Bulk indexed with explicit mapping → verified 814 documents |
| Queries | Demonstrated term, range, and aggregation DSL patterns |